# Generate circuits with GPT

In [1]:
from pathlib import Path
import os
import pickle
from contextlib import nullcontext
import torch
import tiktoken
from nanoGPT.model import GPTConfig, GPT
import pandas as pd
import json
from tqdm import tqdm
import random
import numpy as np
from matplotlib import pyplot as plt
from collections import defaultdict

In [2]:
out_dir = 'nanoGPT/out-qaoa_n12w_092024/'
data_dir = 'nanoGPT/data/qaoa_n12w_092024/'

In [3]:
seed = 1337
init_from = 'resume' # either 'resume' (from an out_dir) or a gpt2 variant (e.g. 'gpt2-xl')
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
compile = False # use PyTorch 2.0 to compile the model to be faster
#exec(open(nanogpt_path.joinpath('configurator.py')).read()) # overrides from command line or config file
# -----------------------------------------------------------------------------

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

In [4]:
# init from a model saved in a specific directory
out_path = Path(out_dir)
ckpt_path = f'{out_dir}/ckpt.pt'
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

/tmp/job_28640055/step_0.0/ipykernel_256135/3885413165.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


number of parameters: 11.52M


<All keys matched successfully>

In [5]:
model.eval()
model.to(device)
print()

In [6]:
meta = pd.read_pickle(f'{data_dir}/meta.pkl')

In [7]:
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [8]:
def extract_graph(token_seq):
    graph_seq = []

    for idx, tok in enumerate(token_seq):
        graph_seq.append(tok)
        if tok == 'end_of_graph':
            break
    adapt_seq = token_seq[idx+1:-1]
    return graph_seq, adapt_seq

## Start generation

In [9]:
num_samples = 5 # number of samples to draw
max_new_tokens = 500 # number of tokens generated in each sample
temperature = 0.1 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability

In [54]:
n_samples_per_batch = 50 # max number of distinct graphs in a batch

### Opening val dataset

In [55]:
sample_size = 1000

In [56]:
all_data_df = pd.read_pickle(
    f'{data_dir}/combined_res_tok_shf_df.pkl'
)

print("Dataset size:", len(all_data_df), 'circuits.')

Dataset size: 100540 circuits.


In [57]:
print(all_data_df['token_seq_round_d2'][264])

['bos', (1, 2), 0.52, (1, 3), 0.58, (2, 3), 0.94, (1, 4), 0.12, (3, 4), 0.46, (3, 5), 0.74, (4, 5), 0.27, (3, 6), 0.77, (4, 6), 0.96, (5, 6), 0.53, (1, 7), 0.35, (2, 7), 0.42, (3, 7), 0.56, (4, 7), 0.42, (5, 7), 0.65, (6, 7), 0.65, (2, 8), 0.88, (3, 8), 0.25, (4, 8), 0.83, (5, 8), 0.51, (6, 8), 0.35, (2, 9), 0.53, (3, 9), 0.07, (4, 9), 0.62, (5, 9), 0.1, (8, 9), 0.94, (1, 10), 0.24, (2, 10), 0.23, (3, 10), 0.67, (4, 10), 0.72, (7, 10), 0.51, (8, 10), 0.54, (9, 10), 0.76, (1, 11), 0.53, (2, 11), 0.06, (4, 11), 0.21, (5, 11), 0.77, (6, 11), 0.52, (7, 11), 0.16, (8, 11), 0.22, (9, 11), 0.49, (10, 11), 0.45, (1, 12), 0.23, (3, 12), 0.92, (4, 12), 0.15, (6, 12), 0.02, (8, 12), 0.05, (9, 12), 0.48, (10, 12), 0.25, 'end_of_graph', 'new_layer_p', 140, -0.79, 0.0, 'new_layer_p', 108, -0.79, 0.0, 'new_layer_p', 113, -0.79, 0.0, 'new_layer_p', 168, -0.79, -0.0, 'new_layer_p', 149, -0.79, -0.0, 'new_layer_p', 185, -0.79, 0.0, 'new_layer_p', 181, -0.79, -0.0, 'new_layer_p', 68, -0.79, 0.0, 'new_lay

In [ ]:
test_df = all_data_df[
    all_data_df['label'] == 'test'
]

token_seq_col = 'token_seq_round_d2'

sampled_idx_list = random.sample(list(test_df.index), sample_size)

test_run_df = test_df.loc[sampled_idx_list]

n_edges_to_count_dict = test_run_df['edgelist_list_len'].value_counts().to_dict()

In [62]:
# Batched inference based on number of edges. 
# We group graphs with the same number of edges together
# such that we can merge them into a tensor to keep the input length size consistent.

adapt_gpt_out_list_dict = defaultdict(list)
x_list_dict = defaultdict(list)
y_dict = dict()

pbar = tqdm(n_edges_to_count_dict.items())

for n_edges, n_graphs in pbar:
    pbar.set_description(f"Current batch: n_edges: {n_edges}, n_graphs: {n_graphs}")
    cur_test_run_df = test_run_df[
        test_run_df['edgelist_list_len'] == n_edges
    ]
    
    for row_idx, graph_df_row in cur_test_run_df.iterrows():
    #graph_df_row = test_df.loc[graph_idx]
        start, adapt_seq = extract_graph(graph_df_row[token_seq_col])
        start_ids = encode(start)
        x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])
        x_list_dict[n_edges].append(x)

        adapt_gpt_out_dict = dict()
        adapt_gpt_out_dict['graph'] = start[1:-1]
        adapt_gpt_out_dict['q_circuits'] = []
        adapt_gpt_out_dict['adapt_circuit'] = adapt_seq
        adapt_gpt_out_dict['adapt_full_ar'] = graph_df_row['approx_ratio']
        adapt_gpt_out_dict['graph_prefix'] = graph_df_row['graph_id']
        adapt_gpt_out_dict['energy_mqlib'] = graph_df_row['energy_mqlib']
        adapt_gpt_out_list_dict[n_edges].append(adapt_gpt_out_dict)
    
    cur_batch_torch = torch.vstack(x_list_dict[n_edges])

    # Calculate total samples and number of mini-batches
    total_samples = cur_batch_torch.size(0)
    n_batches = (total_samples + n_samples_per_batch - 1) // n_samples_per_batch  # Ensure ceiling division

    # Initialize an empty list for results
    y_list = []
    
    # Run inference in mini-batches
    with torch.no_grad():
        for i in tqdm(range(n_batches), desc='Internal batch progress', disable=True):
            start_idx = i * n_samples_per_batch
            end_idx = min((i + 1) * n_samples_per_batch, total_samples)
            mini_batch = cur_batch_torch[start_idx:end_idx]
    
            # Repeat the mini-batch for num_samples
            mini_batch_repeated = mini_batch.repeat(num_samples, 1)
    
            with ctx:
                y = model.generate(
                    mini_batch_repeated,
                    max_new_tokens,
                    temperature=temperature,
                    top_k=top_k
                )
    
            # Collect results from each mini-batch
            y_list.append(y.detach().cpu())
    
    # Concatenate results from all mini-batches
    y_dict[n_edges] = torch.cat(y_list, dim=0)

Current batch: n_edges: 7, n_graphs: 1: 100%|███████████████████████████████████████████████| 53/53 [03:02<00:00,  3.45s/it]


In [63]:
for n_edges, cur_adapt_gpt_out_list in adapt_gpt_out_list_dict.items():
    cur_full_y_tensor = y_dict[n_edges]
    
    for graph_idx in range(len(cur_adapt_gpt_out_list)):
        
        cur_y_tensor = cur_full_y_tensor[graph_idx::len(cur_adapt_gpt_out_list)]
        
        for k in range(num_samples):
            cur_gen_result = decode(cur_y_tensor[k].tolist())
            cur_circ = []
            circ_flag = 0
            for idx, tok in enumerate(cur_gen_result):
                if tok == 'end_of_graph':
                    circ_flag = 1
                if circ_flag:
                    cur_circ.append(tok)
                if tok == 'eos':
                    break
            cur_adapt_gpt_out_list[graph_idx]['q_circuits'].append(cur_circ[1:-1])

In [64]:
adapt_gpt_test_samples_list = []
for n_edges, cur_adapt_gpt_out_list in adapt_gpt_out_list_dict.items():
    adapt_gpt_test_samples_list += cur_adapt_gpt_out_list

## Formatting for `ADAPT.jl`

In [65]:
def circ_sanity_check(cur_q_circ):
    
    lr_sep_list = cur_q_circ[0::4]
    op_idx_list = cur_q_circ[1::4]

    num_vals = cur_q_circ[2::4] + cur_q_circ[3::4]

    if any(
        [type(el) != int for el in op_idx_list]
    ):
        print('wrong op_idx_list')
        return False

    if any(
        [type(el) != str for el in lr_sep_list]
    ):
        print('wrong lr_sep_list')
        return False
    
    if len(cur_q_circ) % 4:
        print('Wrong length')
        return False

    return True

In [66]:
for idx in range(len(adapt_gpt_test_samples_list)):
    q_circ_filt_list = []
    for circ in adapt_gpt_test_samples_list[idx]['q_circuits']:
        filt_flag = circ_sanity_check(circ)
        if not filt_flag:
            print(adapt_gpt_test_samples_list[idx]['graph_prefix'], '\n')
        else:
            q_circ_filt_list.append(circ)

    adapt_gpt_test_samples_list[idx]['q_circuits'] = q_circ_filt_list

for gr_dict in adapt_gpt_test_samples_list:
    graph_jl_list = []

    graph_edges_list = gr_dict['graph'][::2]
    graph_weights_list = gr_dict['graph'][1::2]
    for edge_idx, edge in enumerate(graph_edges_list):
        cur_edge = list(edge)
        cur_edge += [graph_weights_list[edge_idx]]
        graph_jl_list.append(cur_edge)

    gr_dict['graph_w_jl'] = graph_jl_list

In [67]:
adapt_gpt_test_samples_filt_list = []

for rec in adapt_gpt_test_samples_list:
    pos_flag = 1
    if len(rec['adapt_circuit']) % 4:
        pos_flag = 0
    for gpt_circ in rec['q_circuits']:
        if len(gpt_circ) % 4:
            pos_flag = 0
    
    if pos_flag:
        adapt_gpt_test_samples_filt_list.append(rec)

In [68]:
adapt_gpt_test_samples_df = pd.DataFrame(adapt_gpt_test_samples_filt_list)

In [69]:
adapt_gpt_test_samples_df

,graph,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_mqlib,graph_w_jl
0,"[(1, 2), 0.43, (1, 3), 0.99, (1, 4), 0.06, (2,...","[[new_layer_p, 21, -0.79, 0.0, new_layer_p, 45...","[new_layer_p, 21, -0.79, 0.0, new_layer_p, 45,...",1.000000,n_12_worker_r1n31_pid_40925_ts_24-09-21__06_04...,-21.46,"[[1, 2, 0.43], [1, 3, 0.99], [1, 4, 0.06], [2,..."
1,"[(1, 2), 0.49, (2, 3), 0.24, (1, 5), 0.3, (2, ...","[[new_layer_p, 49, -0.79, 0.0, new_layer_p, 37...","[new_layer_p, 49, -0.79, -0.0, new_layer_p, 37...",1.000000,n_12_worker_r1n46_pid_8708_ts_24-09-21__14_14_...,-21.20,"[[1, 2, 0.49], [2, 3, 0.24], [1, 5, 0.3], [2, ..."
2,"[(1, 2), 0.81, (1, 3), 0.88, (2, 3), 0.29, (3,...","[[new_layer_p, 192, -0.79, 0.0, new_layer_p, 2...","[new_layer_p, 257, -0.79, -0.0, new_layer_p, 1...",1.000000,n_12_worker_r1n43_pid_47919_ts_24-09-21__19_45...,-21.19,"[[1, 2, 0.81], [1, 3, 0.88], [2, 3, 0.29], [3,..."
3,"[(1, 2), 0.2, (1, 3), 0.95, (1, 4), 0.92, (3, ...","[[new_layer_p, 165, -0.79, 0.0, new_layer_p, 5...","[new_layer_p, 165, -0.79, 0.0, new_layer_p, 56...",1.000000,n_12_worker_r2l02_pid_15738_ts_24-09-21__14_14...,-20.52,"[[1, 2, 0.2], [1, 3, 0.95], [1, 4, 0.92], [3, ..."
4,"[(1, 2), 0.62, (1, 3), 0.47, (1, 4), 0.83, (2,...","[[new_layer_p, 88, -0.79, 0.0, new_layer_p, 16...","[new_layer_p, 88, -0.79, -0.0, new_layer_p, 48...",1.000000,n_12_worker_r1n46_pid_8701_ts_24-09-21__14_14_...,-21.48,"[[1, 2, 0.62], [1, 3, 0.47], [1, 4, 0.83], [2,..."
...,...,...,...,...,...,...,...
995,"[(2, 3), 0.22, (3, 4), 0.48, (1, 5), 0.13, (2,...","[[new_layer_p, 13, -0.62, 0.44, new_layer_p, 1...","[new_layer_p, 128, -0.79, 0.0, new_layer_p, 13...",0.935780,n_12_worker_r1n25_pid_19444_ts_24-09-22__07_25...,-8.72,"[[2, 3, 0.22], [3, 4, 0.48], [1, 5, 0.13], [2,..."
996,"[(1, 7), 0.98, (5, 8), 0.49, (1, 9), 0.84, (4,...","[[new_layer_p, 37, -0.79, 0.0, new_layer_p, 16...","[new_layer_p, 37, -0.79, -0.0, new_layer_p, 16...",0.883126,n_12_worker_r1n44_pid_21078_ts_24-09-21__19_45...,-7.23,"[[1, 7, 0.98], [5, 8, 0.49], [1, 9, 0.84], [4,..."
997,"[(1, 2), 0.84, (1, 3), 0.15, (2, 3), 0.89, (1,...","[[new_layer_p, 117, -0.79, 0.0, new_layer_p, 6...","[new_layer_p, 116, -0.79, 0.0, new_layer_p, 80...",1.000000,n_12_worker_r1n09_pid_10694_ts_24-09-21__12_02...,-34.86,"[[1, 2, 0.84], [1, 3, 0.15], [2, 3, 0.89], [1,..."
998,"[(1, 2), 0.61, (1, 3), 0.23, (2, 3), 0.93, (1,...","[[new_layer_p, 161, -0.79, 0.0, new_layer_p, 5...","[new_layer_p, 160, -0.79, 0.0, new_layer_p, 24...",1.000000,n_12_worker_r1n43_pid_23448_ts_24-09-21__14_14...,-33.72,"[[1, 2, 0.61], [1, 3, 0.23], [2, 3, 0.93], [1,..."


## Saving

In [70]:
adapt_gpt_circ_out_fpath = 'data/adapt_gpt_n12w_batched_092024_d2_test_out.json'

In [71]:
with open(adapt_gpt_circ_out_fpath, 'w') as f:
    f.write(json.dumps(adapt_gpt_test_samples_filt_list))